# Multiperiod Workspace Scenarios
This notebook moves from the scalar teaching cases to two real multiperiod derivatives of shipped examples: a direct-process crude preheat train and a seasonal site-style utility case.

**Support notice:** This advanced notebook exercises unsupported internal owner modules. Only `from OpenPinch.main import pinch_analysis_service` is compatibility protected.


## Case context
The two packaged multiperiod inputs stay recognizable descendants of the original examples. `crude_preheat_train_multiperiod.json` is still a direct process case, while `zonal_site_multiperiod.json` keeps the site-style utility picture and adds seasonal demand and recovery variation.


In [ ]:
import pandas as pd

from OpenPinch.application.problem import PinchProblem


## Direct multiperiod targeting on the crude preheat train
Start by reading the named `period_ids` and the case weights carried by the case input. The crude derivative uses the named periods `turndown`, `base`, and `peak`. Then solve one direct target per period so the utility and pinch differences are explicit.


In [ ]:
crude_problem = PinchProblem(
    "crude_preheat_train_multiperiod.json",
    project_name="crude_multiperiod",
)
crude_case_input = crude_problem.to_problem_json()

{
    "period_ids": crude_problem.period_ids,
    "weights": dict(zip(crude_problem.period_ids, crude_problem.master_zone.weights)),
}


In [ ]:
pd.DataFrame(
    [
        {
            "stream": stream["name"],
            "t_target_values": stream.get("t_target", {}).get("values"),
            "t_supply_values": stream.get("t_supply", {}).get("values"),
            "heat_flow_values": stream.get("heat_flow", {}).get("values"),
        }
        for stream in crude_case_input["streams"]
        if stream["name"] in [
            "Desalted crude to preheat train",
            "Kerosene side-draw preheat",
            "Atmospheric residue pumparound",
            "Diesel pumparound return",
        ]
    ]
)


In [ ]:
crude_period_targets = {
    period_id: crude_problem.target.direct_heat_integration(period_id=period_id)
    for period_id in crude_problem.period_ids
}

pd.DataFrame(
    [
        {
            "period_id": period_id,
            "weight": crude_problem.master_zone.weights[idx],
            "Qh": target.hot_utility_target,
            "Qc": target.cold_utility_target,
            "Qr": target.heat_recovery_target,
            "hot_pinch": target.hot_pinch,
            "cold_pinch": target.cold_pinch,
        }
        for idx, (period_id, target) in enumerate(crude_period_targets.items())
    ]
)


In [ ]:
crude_problem.target.direct_heat_integration(period_id="peak")
crude_problem.summary_frame()[
    ["Target", "Period ID", "Hot Utility Target", "Cold Utility Target", "Hot Pinch", "Cold Pinch"]
]


In [ ]:
crude_all_periods = crude_problem.target_all_periods(parallel=False)

pd.DataFrame(
    [
        {
            "period_id": period_id,
            "Qh": next(target for target in output.targets if target.name.endswith("/Direct Integration")).Qh,
            "Qc": next(target for target in output.targets if target.name.endswith("/Direct Integration")).Qc,
            "Qr": next(target for target in output.targets if target.name.endswith("/Direct Integration")).Qr,
            "hot_pinch": next(target for target in output.targets if target.name.endswith("/Direct Integration")).pinch_temp.hot_temp,
            "cold_pinch": next(target for target in output.targets if target.name.endswith("/Direct Integration")).pinch_temp.cold_temp,
        }
        for period_id, output in crude_all_periods.items()
    ]
)


## Seasonal indirect targeting on the site-style case
The seasonal derivative keeps the same utility-system structure, but the residential and hospital loads now move by season. Its named periods are `summer`, `shoulder`, and `winter`. Solve one full period first, then use the focused `indirect_heat_integration(period_id=...)` accessor on that named period.


In [ ]:
site_problem = PinchProblem(
    "zonal_site_multiperiod.json",
    project_name="seasonal_site",
)
site_case_input = site_problem.to_problem_json()

{
    "period_ids": site_problem.period_ids,
    "weights": dict(zip(site_problem.period_ids, site_problem.master_zone.weights)),
}


In [ ]:
pd.DataFrame(
    [
        {
            "zone": stream["zone"],
            "stream": stream["name"],
            "t_target_values": stream.get("t_target", {}).get("values"),
            "t_supply_values": stream.get("t_supply", {}).get("values"),
            "heat_flow_values": stream.get("heat_flow", {}).get("values"),
        }
        for stream in site_case_input["streams"]
        if (stream["zone"], stream["name"]) in [
            ("Residential[S]", "Space heating"),
            ("Residential[S]", "Hot water 2"),
            ("Hospital[S]", "Heating"),
            ("Hospital[S]", "Swimming pool water"),
            ("Chemical Plant[S]", "A2"),
            ("Food Plant[S]", "B1"),
        ]
    ]
)


In [ ]:
site_problem.target(period_id="winter")
winter_total_site = site_problem.target.indirect_heat_integration(period_id="winter")

{
    "target": winter_total_site.name,
    "period_id": winter_total_site.period_id,
    "Qh": winter_total_site.hot_utility_target,
    "Qc": winter_total_site.cold_utility_target,
    "Qr": winter_total_site.heat_recovery_target,
    "hot_pinch": winter_total_site.hot_pinch,
    "cold_pinch": winter_total_site.cold_pinch,
}


In [ ]:
site_problem.summary_frame()[
    ["Target", "Period ID", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"]
]


In [ ]:
site_all_periods = site_problem.target_all_periods(parallel=False)

pd.DataFrame(
    [
        {
            "period_id": period_id,
            "weight": site_problem.master_zone.weights[idx],
            "site_direct_Qc": next(target for target in output.targets if target.name.endswith("/Direct Integration")).Qc,
            "site_total_site_Qc": next(target for target in output.targets if target.name.endswith("/Total Site Target")).Qc,
            "site_total_site_Qr": next(target for target in output.targets if target.name.endswith("/Total Site Target")).Qr,
            "site_total_site_hot_pinch": next(target for target in output.targets if target.name.endswith("/Total Site Target")).pinch_temp.hot_temp,
        }
        for idx, (period_id, output) in enumerate(site_all_periods.items())
    ]
)


## Interpretation
The crude case shows how named periods can move both utility targets and the pinch location on a single process. The seasonal site case shows a different use of the same multiperiod machinery: the Total Site answer changes because winter demand absorbs more of the plant heat surplus, so the cross-period table becomes the right place to interpret utility targets before you average anything with the stored weights.
